# Spark 上手教程 (hands-on)

基于本项目真实数据 (`/data/yizhou/output/*.parquet`)。**在 exx 上运行**
(VSCode Remote-SSH 连 exx), kernel 选 `/data/yizhou/envs/spark`。

按顺序从上往下运行每个 cell。Spark UI: `http://localhost:4040`
(本地看: 另开终端 `ssh -L 4040:localhost:4040 exx-server`)。

## Setup — 建 SparkSession
notebook 里没有自动的 `spark`, 要自己建 (和我们脚本里一样)。学习用 `local[4]`+8g。

In [ ]:
import os, time, sys
os.environ["SPARK_LOCAL_DIRS"] = "/data/yizhou/spark-tmp"
sys.path.append("/data/yizhou/repo")          # 让 Lesson 7 的 `from spark_common import ...` 能用
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
    .master("local[4]")
    .appName("tutorial")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
print("Spark", spark.version, "| master", sc.master, "| UI http://localhost:4040")

## Lesson 0 — driver / executor / partition / task
数据被切成 **partition**; 每个 partition 上一段计算 = 一个 **task**; **executor** 跑 task,
**driver** 指挥。`local[4]` = 本地 4 线程当 executor, 一次最多同时跑 4 个 task。

In [ ]:
print("master:", sc.master)
print("defaultParallelism:", sc.defaultParallelism)   # 4
df = spark.read.parquet("/data/yizhou/output/reviews_clean.parquet")
print("这份数据的分区数:", df.rdd.getNumPartitions())
# 思考: 53 个分区 / 一次跑 4 个 → 大约分 53/4 ≈ 14 波跑完。分区数决定并行度上限。

## Lesson 1 — DataFrame 基础: 读 / schema / 看数据
DataFrame = 带 schema 的分布式表, 背后有 Catalyst 优化器。

In [ ]:
df = spark.read.parquet("/data/yizhou/output/reviews_clean.parquet")
df.printSchema()
df.show(5, truncate=40)
print("列:", df.columns)

## Lesson 2 — 惰性求值: transformation vs action ⭐最重要
**transformation** (filter/select/join/groupBy...) 是懒的, 只搭计划;
**action** (count/show/collect/write...) 才真正触发计算。

In [ ]:
# transformation: 几乎 0 秒 (什么都没算, 只搭 DAG)
t = time.time()
beauty = df.filter(df.category == "Beauty_and_Personal_Care").filter(df.rating >= 4.0)
print("定义 transformation:", round(time.time()-t, 4), "秒")

# action: 现在才真跑
t = time.time()
n = beauty.count()
print("count =", n, " | action 用时:", round(time.time()-t, 2), "秒")

**面试金句**: "Spark 惰性求值 —— transformation 只搭 DAG, action 才执行。这正是我们
k-core 循环不 cache 会爆炸的原因: 每轮 count 都从头重算整个 DAG。"（Lesson 5 会亲手复现）

## Lesson 3 — .explain() 读执行计划: narrow vs wide
narrow (filter/select) 不搬数据; wide (groupBy/join/window) 需要 **shuffle** (计划里的 `Exchange`)。

In [ ]:
print("===== filter (narrow, 无 Exchange) =====")
beauty.explain()
print("\n===== groupBy (wide, 有 Exchange = shuffle) =====")
df.groupBy("category").count().explain()

## Lesson 4 — shuffle 后的分区数
每次 shuffle 后分区数 = `spark.sql.shuffle.partitions`。太少并行不够, 太多小文件多。

In [ ]:
print("当前 shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
g = df.groupBy("category").count()
g.show()
print("groupBy(parent_asin) 之后的分区数:", df.groupBy("parent_asin").count().rdd.getNumPartitions())

## Lesson 5 — cache: 亲手复现我们的 k-core 教训 (缩小版) ⭐
不 cache: 每个 action 都重算整条血缘。cache: 第一次物化, 之后读内存。

In [ ]:
mid = df.filter(df.verified_purchase == True).groupBy("user_id").count()

print("--- 不 cache (两次都重算) ---")
t = time.time(); mid.count();                    print("count1:", round(time.time()-t,2), "s")
t = time.time(); mid.filter("count>=5").count(); print("count2:", round(time.time()-t,2), "s")

mid.cache()
print("--- cache 后 ---")
t = time.time(); mid.count();                    print("count1 (首次物化):", round(time.time()-t,2), "s")
t = time.time(); mid.filter("count>=5").count(); print("count2 (复用内存):", round(time.time()-t,2), "s")
mid.unpersist()

**面试金句 (头号 deliverable)**: "迭代算法 惰性+不cache = 每轮重算整个血缘。我实测 k-core
uncached 第6轮单轮188s (cached仅12s, 15×), 第7轮跑爆; 加 `.cache()` 后每轮拉平到 ~11s。"

## Lesson 6 — 分区与数据倾斜 (skew)
item 的交互是幂律 (少数极热); 但按 key 重分区后行级倾斜取决于"最热 key 有没有大过一个分区"。

In [ ]:
five = spark.read.parquet("/data/yizhou/output/reviews_5core.parquet")
ic = five.groupBy("parent_asin").count()
ic.selectExpr("percentile_approx(count, array(0.5,0.95,0.99), 10000) as p50_95_99",
              "max(count) as max").show(truncate=False)

def part_rows(dfx, key, nparts=64):
    return (dfx.repartition(nparts, key)
              .withColumn("pid", F.spark_partition_id()).groupBy("pid").count()
              .selectExpr("min(count) mn", "avg(count) avg", "max(count) mx", "stddev(count) sd"))
print("按 user_id 分区 (预期均衡):"); part_rows(five, "user_id").show()
print("按 parent_asin 分区 (预期更倾斜):"); part_rows(five, "parent_asin").show()

**面试金句**: "倾斜取决于分区键。user 侧均衡 (无鲸鱼用户), 幂律在 item 侧; 但这个规模最热
key 仍没大过一个分区, 所以行级倾斜只 1.18×。这也解释了 AQE 为何没收益。"

## Lesson 7 — Join: broadcast vs sort-merge
大表 ⋈ 小表: broadcast 小表到每个 task, 免掉对大表的 shuffle。但小表必须真的小。

In [ ]:
from spark_common import META_SCHEMA, meta_path
meta = (spark.read.schema(META_SCHEMA).json(meta_path("Toys_and_Games"))
        .dropDuplicates(["parent_asin"]))
five = spark.read.parquet("/data/yizhou/output/reviews_5core.parquet")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # 关自动 broadcast
print("===== 强制 SortMergeJoin (看两个 Exchange + SortMergeJoin) =====")
five.join(meta, "parent_asin", "left").explain()
print("\n===== 显式 broadcast (看 BroadcastHashJoin) =====")
five.join(F.broadcast(meta), "parent_asin", "left").explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10*1024*1024))  # 调回默认

**面试金句 (我们的意外)**: "天真 broadcast 整张 metadata (3.5M行) 会失败 (>1GB 超
maxResultSize, 它不是小表); 过滤到 715K 才行, 之后比 SortMergeJoin 快 1.37×。"

## Lesson 8 — Window: sessionize 与 leave-one-out
`Window.partitionBy(user).orderBy(ts)` = 每个用户内部按时间排序 → 序列推荐的序列。

In [ ]:
five = spark.read.parquet("/data/yizhou/output/reviews_5core.parquet")
w_asc  = Window.partitionBy("user_id").orderBy(F.col("ts_ms").asc(),  F.col("parent_asin").asc())
w_desc = Window.partitionBy("user_id").orderBy(F.col("ts_ms").desc(), F.col("parent_asin").desc())
labeled = (five.withColumn("pos", F.row_number().over(w_asc))
                .withColumn("rev", F.row_number().over(w_desc))
                .withColumn("split", F.when(F.col("rev")==1,"test")
                                      .when(F.col("rev")==2,"valid").otherwise("train")))
print("train/valid/test 行数:"); labeled.groupBy("split").count().show()

# 挑一个有 6~10 次交互的用户, 看他的完整序列 (pos=时间正序, split=切分)
u = five.groupBy("user_id").count().filter("count between 6 and 10").limit(1).collect()[0]["user_id"]
print("示例用户:", u)
labeled.filter(F.col("user_id")==u).orderBy("pos").select("pos","rev","parent_asin","split").show(truncate=False)

## Lesson 9 — 读 Spark UI
先跑下面一个 shuffle 作业, 然后浏览器开 `http://localhost:4040`:
- **Jobs**: 一个 action = 一个 job, 拆成几个 stage。
- **Stages**: stage 边界=shuffle 边界; 看 **Shuffle Read/Write** 大小。
- **Stage 内 Tasks**: 看 **task duration 分布**, 少数特别慢 = straggler = 倾斜。
- **SQL**: 看物理计划图 (Exchange / BroadcastHashJoin / SortMergeJoin)。

In [ ]:
# 跑一个明显的 shuffle 作业, 然后去 UI 的 Stages 页看这次 shuffle read/write 和 task 时间分布
res = five.groupBy("parent_asin").count().count()
print("distinct items:", res, "-> 现在去 http://localhost:4040 的 Stages 页看这次作业")

## Lesson 10 — 写 Parquet (分区输出)
Parquet=列式存储 (只读需要的列+压缩); `partitionBy` 按值分子目录, 之后过滤能"分区裁剪"。

In [ ]:
(labeled.select("user_id","parent_asin","ts_ms","split")
        .write.mode("overwrite").partitionBy("split")
        .parquet("/data/yizhou/output/_tutorial_out.parquet"))
import subprocess
print(subprocess.run(["ls","/data/yizhou/output/_tutorial_out.parquet"],
                     capture_output=True, text=True).stdout)
# 读回来 + 分区裁剪 (只读 split=test 的子目录)
spark.read.parquet("/data/yizhou/output/_tutorial_out.parquet").filter("split='test'").count()

## 收尾: 关掉 SparkSession

In [ ]:
spark.stop()
print("done")

---
### 一句话面试叙事
"我把 5.9 千万条亚马逊评论用 PySpark 做成序列推荐训练集。40GB 能装进 250G 内存, 所以选 local
模式避免集群 shuffle 开销, 把时间花在真瓶颈: 实测不 cache 的迭代 k-core 因惰性求值每轮重算整个
DAG 而爆炸(15×), 加 cache 拉平; 量化数据倾斜发现它取决于分区键、这个规模下温和, 也解释了 AQE
无收益; 发现'元数据小表'其实>1GB broadcast 会失败, 过滤后才 1.37× 提速。全部有数字、有物理计划。"